### **Protein Structural Analysis Pipeline**


An elaborate pipeline for analyzing protein structures obtained from AlphaFold.:
1. Loading and RMSD-based Selection: Extracting AlphaFold models from a ZIP archive and identifying the most representative model.
2. Post-Translational Modification (PTM) with gemmi: Programmatically modifying specific residues (e.g., tryptophan oxidation).
3. Visualization: Interactive 3D visualization of the protein structures.
4. Solvent Accessible Surface Area (SASA) Analysis: Quantifying the surface accessibility of specific residues.
5. Proximity Analysis: Identifying residues in close spatial contact.
The results from SASA and Proximity analyses are saved to CSV files.

In [ ]:
# Install necessary libraries
!pip install openmm py3Dmol gemmi freesasa biopython
pip install openpyxl

# Core Python libraries
import zipfile
import os
import itertools
import tempfile
from collections import Counter
import warnings # Added for warning suppression

# Scientific computing libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Structural biology libraries
import gemmi
import freesasa
from Bio.PDB import MMCIFParser, Superimposer, PDBIO, MMCIFIO, Atom, NeighborSearch, Selection, PDBParser
from Bio.PDB.PDBParser import PDBConstructionWarning # Added for warning suppression

#Visualization tool
import py3Dmol
import os

# Suppress specific Biopython warnings that are non-critical, as they don't affect results
warnings.filterwarnings('ignore', category=PDBConstructionWarning)

print("All necessary libraries installed and imported.")

## 2 and 3. Load AlphaFold Data, RMSD Analysis and addition of oxygen in the CH2 atom of tryptophan 47





This section extracts AlphaFold models from a ZIP file, calculates pairwise RMSD between models, and selects the model with the lowest RMSD as the 'best' representative for further analysis. This 'best' model is then saved as a CIF file.Further,this section defines a function to add an oxygen atom to specific tryptphan residues to simulate oxidation. This modification is applied to the selected best model, and the modified structure is saved to a new CIF file.

In [ ]:
# The lowest RMSD PTM model identified from previous steps:
cif_file_to_analyze_ptm = '/content/protein_structures_ptm_analysis/fold_lrp8_vl_ptm_n_d_model_2.cif'

# 1. Load the selected PTM structure using gemmi
ptm_structure_to_modify = gemmi.read_structure(cif_file_to_analyze_ptm)
print(f"Loading structure from: {cif_file_to_analyze_ptm}")
print("Structure loaded successfully with gemmi.")

# Define target residue and chain
target_res_id = 47
target_chain_id = 'A'
res_name = 'TRP' # Tryptophan
target_atom_name = 'CH2' # The carbon atom to oxidize
new_oxygen_atom_name = 'OXT' # Name for the new oxygen atom

found_trp_residue = False
modified_residue = None
ch2_atom_to_oxidize = None

# 2. Find TRP47 in Chain A and its CH2 atom using gemmi objects
for model in ptm_structure_to_modify:
    for chain in model:
        if chain.name.strip() == target_chain_id:
            for residue in chain:
                if residue.name.strip() == res_name and residue.seqid.num == target_res_id:
                    modified_residue = residue
                    for atom in residue:
                        if atom.name.strip() == target_atom_name:
                            ch2_atom_to_oxidize = atom
                            found_trp_residue = True
                            break
                    break # Found the target residue
            break # Found the target chain
    if found_trp_residue: # Exit model loop if found
        break

if not found_trp_residue or not ch2_atom_to_oxidize:
    print(f"Error: {res_name}{target_res_id} (Chain {target_chain_id}) or its {target_atom_name} atom not found in the selected PTM structure. Skipping modification.")
else:
    print(f"Found {res_name}{target_res_id} ({target_atom_name} atom) in Chain {target_chain_id}.")

    # 3. Calculate position for the new oxygen atom
    CO_BOND_LENGTH = 1.43  # Typical C-O single bond length in Å

    # CH2 is typically bonded to CZ2 and CZ3 in the indole ring.
    # We'll place the new oxygen atom extending outwards from the CH2.
    cz2_atom = None
    cz3_atom = None
    for atom in modified_residue:
        if atom.name.strip() == 'CZ2':
            cz2_atom = atom
        elif atom.name.strip() == 'CZ3':
            cz3_atom = atom

    if cz2_atom and cz3_atom:
        ch2_coord = np.array(ch2_atom_to_oxidize.pos.tolist())
        cz2_coord = np.array(cz2_atom.pos.tolist())
        cz3_coord = np.array(cz3_atom.pos.tolist())

        # Calculate vectors from neighbors to CH2 to determine an outward direction
        vec_cz2_ch2 = ch2_coord - cz2_coord
        vec_cz3_ch2 = ch2_coord - cz3_coord

        direction_vector = vec_cz2_ch2 + vec_cz3_ch2
        unit_direction = direction_vector / np.linalg.norm(direction_vector)

        # Place the new oxygen atom along this direction at a typical C-O bond length
        new_o_coord = ch2_coord + unit_direction * CO_BOND_LENGTH

        # 4. Create and add a new gemmi.Atom object
        new_o_atom = gemmi.Atom()
        new_o_atom.name = new_oxygen_atom_name
        new_o_atom.element = gemmi.Element("O")
        new_o_atom.pos.x = new_o_coord[0]
        new_o_atom.pos.y = new_o_coord[1]
        new_o_atom.pos.z = new_o_coord[2]
        new_o_atom.charge = 0
        new_o_atom.occ = 1.0
        new_o_atom.b_iso = 20.0 # Default B-factor

        # Add the new atom to the residue
        modified_residue.add_atom(new_o_atom)
        print(f"Added new oxygen atom '{new_o_atom.name}' to {res_name}{target_res_id} at ({new_o_atom.pos.x:.3f}, {new_o_atom.pos.y:.3f}, {new_o_atom.pos.z:.3f}).")

        # 5. Save the modified structure to a new CIF file
        modified_ptm_cif_path = '/content/modified_ptm_structure.cif'
        ptm_structure_to_modify.make_mmcif_document().write_file(modified_ptm_cif_path)
        print(f"Modified PTM structure saved to: {modified_ptm_cif_path}")

    else:
        print("Error: Could not find CZ2 or CZ3 atoms in TRP47 for calculating oxygen placement direction.")

### 3. Visulization of the PTM modified structure

This section visualizes the original best model with target residues highlighted, and then the modified structure with the added oxygen atoms clearly marked, along with other highlighted residues for context.

In [ ]:
 # 6. Visualize the modified structure, highlighting the change
        view_mod = py3Dmol.view(width=800, height=600)
        view_mod.setBackgroundColor('white')

        # Add modified PTM structure (Model 0 in this case) in blue
        modified_ptm_content_for_viz = open(modified_ptm_cif_path, 'r').read()
        view_mod.addModel(modified_ptm_content_for_viz, 'cif')
        view_mod.setStyle({'model': 0}, {'cartoon': {'color': 'blue'}})
        # Highlight modified TRP47 as red sticks
        view_mod.addStyle({'model': 0, 'resi': target_res_id, 'chain': target_chain_id}, {'stick': {'colorscheme': 'redCarbon'}})
        # Show the new oxygen atom as an orange sphere
        view_mod.addStyle({'model': 0, 'resi': target_res_id, 'chain': target_chain_id, 'atom': new_oxygen_atom_name}, {'sphere': {'radius': 0.5, 'color': 'orange'}})
        # Label for modified PTM
        view_mod.addLabel('Modified PTM Structure (Blue)', {'position': {'x': float(np.mean(new_o_coord[0] + 5)), 'y': float(np.mean(new_o_coord[1])), 'z': float(np.mean(new_o_coord[2]))}, 'fontColor': 'blue', 'backgroundColor': 'white', 'fontSize': 12, 'showBackground': True})
        # Label for W47 Oxidation
        view_mod.addLabel(f'W{target_res_id} CH2 Oxidation ({new_oxygen_atom_name} Orange Sphere)', {'position': {'x': float(new_o_coord[0]), 'y': float(new_o_coord[1]), 'z': float(new_o_coord[2])}, 'fontColor': 'orange', 'backgroundColor': 'white', 'fontSize': 12, 'showBackground': True})

        view_mod.zoomTo()
        view_mod.show()


### 4. Detailed verification of the added OXYGEN in tryptophan 47

In [ ]:
 # 1. Check the actual coordinates of your added Oxygen and CH2
ch2_atom = None
oxt_atom = None

# Ensure CO_BOND_LENGTH is defined, assuming it's available from a previous cell's execution
# If not, it should be defined here or sourced from the modification cell.
# For now, let's assume it's globally available or define it locally for safety.
CO_BOND_LENGTH = 1.43  # Typical C-O single bond length in Å (Duplicated for robustness, assuming it's defined in 612f55b9)

for model in ptm_structure_to_modify: # Use the modified structure here (assuming ptm_structure_to_modify is the oxidized one)
    for chain in model:
        for res in chain:
            if res.seqid.num == 47 and chain.name == 'A':
                for atom in res:
                    if atom.name == "OXT":
                        oxt_atom = atom
                        print(f"DEBUG: OXT Position is {atom.pos}")
                    if atom.name == "CH2":
                        ch2_atom = atom
                        print(f"DEBUG: CH2 Position is {atom.pos}")

# 2. Calculate the distance between OXT and CH2 if both are found
if ch2_atom and oxt_atom:
    distance = ch2_atom.pos.dist(oxt_atom.pos)
    print(f"\nDEBUG: Distance between CH2 and OXT is {distance:.3f} Å")
    # The CO_BOND_LENGTH was 1.43, so we can compare to that.
    if abs(distance - CO_BOND_LENGTH) > 0.5: # Allow for some deviation
        print("WARNING: The OXT atom might be unexpectedly far from CH2. Expected bond length around 1.43 Å.")
    else:
        print("The OXT atom appears to be at a reasonable distance from CH2.")
else:
    print("ERROR: Could not find both CH2 and OXT atoms in TRP47 Chain A for distance calculation.")


### 5. SASA analysis of native and PTM modified tryptphan





In [ ]:
from Bio.PDB import MMCIFParser, PDBIO # Added PDBIO
import os
import tempfile # Added tempfile

def get_residue_sasa(structure_path, chain_id, res_id, atom_names=None):
    """
    Calculates SASA for specific atoms within a residue using freesasa.
    Attempts to convert CIF to PDB for freesasa compatibility if direct loading fails.
    """
    temp_pdb_path = None
    freesasa_structure = None

    try:
        # Attempt direct freesasa loading
        freesasa_structure = freesasa.Structure(structure_path)
        if freesasa_structure.nAtoms() == 0:
            raise Exception(f"freesasa loaded '{structure_path}' directly but found 0 atoms.")

    except Exception as e_freesasa:
        print(f"Warning: freesasa.Structure failed to directly load '{structure_path}'. Attempting Biopython conversion to PDB. Original freesasa error: {e_freesasa}")
        try:
            # Parse CIF with Biopython
            parser = MMCIFParser(QUIET=True)
            biopython_structure = parser.get_structure('temp', structure_path)

            # Check if Biopython structure is empty
            if not any(True for _ in biopython_structure.get_atoms()): # Check if any atoms exist
                raise Exception(f"Biopython parser returned an empty structure for '{structure_path}'. Cannot convert to PDB.")

            # Create a temporary PDB file
            fd, temp_pdb_path = tempfile.mkstemp(suffix=".pdb")
            os.close(fd) # Close the file descriptor immediately

            io = PDBIO()
            io.set_structure(biopython_structure)
            io.save(temp_pdb_path)
            print(f"  Biopython successfully converted '{structure_path}' to temporary PDB: '{temp_pdb_path}'.")

            # Attempt freesasa loading from the temporary PDB
            freesasa_structure = freesasa.Structure(temp_pdb_path)
            if freesasa_structure.nAtoms() == 0:
                raise Exception(f"freesasa loaded temporary PDB '{temp_pdb_path}' but found 0 atoms.")
            print(f"  freesasa successfully loaded '{temp_pdb_path}' with {freesasa_structure.nAtoms()} atoms.")

        except Exception as e_biopython:
            print(f"Error: Biopython conversion to PDB or subsequent freesasa loading from PDB failed for '{structure_path}'. Biopython/freesasa error: {e_biopython}")
            # Ensure the exception is re-raised to stop execution if there's a hard error
            raise e_biopython

    if not freesasa_structure or freesasa_structure.nAtoms() == 0:
        raise Exception(f"Failed to load structure from '{structure_path}' for SASA calculation or structure contains 0 atoms.")

    # Run freesasa calculation
    result = freesasa.calc(freesasa_structure, freesasa.Parameters())

    sasa_sum = 0.0
    atom_sasa = {}

    # Debugging: Print all atoms freesasa sees for the target residue and chain
    print(f"\n--- Debugging freesasa structure for {structure_path} ---")
    found_target_residue_atoms_in_freesasa = False

    # Collect all unique chain IDs and residue numbers to aid debugging
    all_chain_ids = set()
    all_res_nums = set()

    for i in range(freesasa_structure.nAtoms()):
        atom_obj_name = freesasa_structure.atomName(i)
        atom_obj_resname = freesasa.structure.residueName(i)
        atom_obj_resnum = freesasa.structure.residueNumber(i)
        atom_obj_chainlabel = freesasa.structure.chainLabel(i)

        all_chain_ids.add(atom_obj_chainlabel)
        all_res_nums.add(atom_obj_resnum)

        # Corrected comparison: strip whitespace and convert residue number to int
        if atom_obj_chainlabel.strip() == chain_id.strip() and int(atom_obj_resnum.strip()) == res_id:
            found_target_residue_atoms_in_freesasa = True
            print(f"  Found atom {atom_obj_name} in {atom_obj_resname}{atom_obj_resnum} Chain {atom_obj_chainlabel} (Index: {i})")
            if atom_names is None or atom_obj_name in atom_names:
                sasa_sum += result.atomArea(i)
                atom_sasa[atom_obj_name] = result.atomArea(i)

    if not found_target_residue_atoms_in_freesasa:
        print(f"  No atoms matching Chain '{chain_id}', Residue '{res_id}' found in freesasa structure.")
        print(f"  Available Chain IDs: {sorted(list(all_chain_ids))}")
        print(f"  Available Residue Numbers: {sorted(list(all_res_nums))}")

    print(f"--- End Debugging freesasa structure for {structure_path} ---")

    # Clean up temporary PDB file if it was created
    if temp_pdb_path and os.path.exists(temp_pdb_path):
        os.remove(temp_pdb_path)

    return sasa_sum, atom_sasa

print("\n--- Solvent Accessible Surface Area (SASA) Analysis ---")

# SASA for native W47 CH2 atom
native_w47_ch2_sasa, _ = get_residue_sasa(cif_file_to_analyze_native, 'A', 47, atom_names=['CH2'])
print(f"Native W47 (CH2 atom) SASA: {native_w47_ch2_sasa:.3f} Å²")

# SASA for oxidized W47 (including OXT atom)
oxidized_w47_sasa_all_atoms, oxidized_w47_atom_sasa = get_residue_sasa(modified_ptm_cif_path, 'A', 47)
print(f"Oxidized W47 (all atoms) SASA: {oxidized_w47_sasa_all_atoms:.3f} Å²")

# SASA for oxidized W47 CH2 atom
oxidized_w47_ch2_sasa = oxidized_w47_atom_sasa.get('CH2', 0.0)
print(f"Oxidized W47 (CH2 atom only) SASA: {oxidized_w47_ch2_sasa:.3f} Å²")

# SASA for the newly added OXT atom in oxidized W47
oxidized_w47_oxt_sasa = oxidized_w47_atom_sasa.get('OXT', 0.0)
print(f"Oxidized W47 (OXT atom only) SASA: {oxidized_w47_oxt_sasa:.3f} Å²")

### 6. Proximity analysis of the tryptophan 47


In [ ]:
import pandas as pd # Import pandas for data manipulation

# Define the search radius
RADIUS_LIMIT = 4.0  # Angstroms (Changed from 3.2 to 4.0)

# --- 1. Load structures using gemmi ---
native_gemmi_st = gemmi.read_structure(cif_file_to_analyze_native)
# Corrected path for the oxidized structure containing W47-OXT
# Use the globally defined variable modified_ptm_cif_path
oxidized_gemmi_st = gemmi.read_structure(modified_ptm_cif_path)

# --- 2. Find W47 and its relevant atoms ---

def find_atom_in_structure(structure, chain_id, res_id, atom_name):
    print(f"DEBUG: Searching for {atom_name} in Chain {chain_id}, Res {res_id} in structure {structure.name}...")
    found_chain = False
    for model in structure:
        for chain in model:
            if chain.name.strip() == chain_id.strip():
                found_chain = True
                print(f"DEBUG: Found Chain '{chain.name.strip()}'. Listing residues in this chain:")
                found_residue = False
                for residue in chain:
                    current_res_id = residue.seqid.num
                    current_res_name = residue.name.strip()
                    print(f"  Residue: {current_res_name}{current_res_id}")
                    if current_res_id == res_id:
                        found_residue = True
                        print(f"DEBUG: Found target Residue {current_res_name}{current_res_id}. Listing atoms:")
                        for atom in residue:
                            current_atom_name = atom.name.strip()
                            print(f"    Atom: {current_atom_name}")
                            if current_atom_name == atom_name.strip():
                                print(f"DEBUG: Found atom {current_atom_name} in Chain {chain.name.strip()}, Res {current_res_name}{current_res_id}.")
                                return atom
                        print(f"DEBUG: Atom {atom_name.strip()} not found in target Residue {current_res_name}{current_res_id}.")
                        break # Exit residue loop once target residue is processed
                if not found_residue:
                    print(f"DEBUG: Target Residue {res_id} not found in Chain {chain.name.strip()}.")
                break # Exit chain loop once target chain is processed

    if not found_chain:
        print(f"DEBUG: Target Chain {chain_id.strip()} not found in structure {structure.name}.")
    print(f"DEBUG: Atom {atom_name.strip()} not found in Chain {chain_id.strip()}, Res {res_id}.")
    return None

# Initialize a list to store all proximity data
all_proximity_data = []

# --- Proximity Analysis for Native W47 (CH2) ---
native_w47_ch2 = find_atom_in_structure(native_gemmi_st, 'A', 47, 'CH2') # Changed from CZ3 to CH2

print("\n--- Neighbors for Native W47 (CH2) ---")
if native_w47_ch2:
    native_search = gemmi.NeighborSearch(native_gemmi_st, RADIUS_LIMIT)
    native_neighbors = list(native_search.find_neighbors(native_w47_ch2, max_dist=RADIUS_LIMIT))
    if native_neighbors:
        for neighbor in native_neighbors:
            atom = neighbor.atom
            if atom and atom.residue:
                print(f"  Atom: {atom.name}, Residue: {atom.residue.name}{atom.residue.seqid.num} Chain: {atom.residue.chain.name}, Distance: {neighbor.dist:.3f} Å")
                all_proximity_data.append({
                    'Structure Type': 'Native',
                    'Target Atom': 'CH2',
                    'Target Residue': 'W47',
                    'Neighbor Atom': atom.name,
                    'Neighbor Residue': f"{atom.residue.name}{atom.residue.seqid.num}",
                    'Neighbor Chain': atom.residue.chain.name,
                    'Distance': neighbor.dist
                })
    else:
        print(f"  No neighbors found within {RADIUS_LIMIT}Å of Native W47 CH2.")
else:
    print("  Native W47 CH2 atom not found.")

# --- Proximity Analysis for Oxidized W47 (CH2) ---
oxidized_w47_ch2 = find_atom_in_structure(oxidized_gemmi_st, 'A', 47, 'CH2')

print("\n--- Neighbors for Oxidized W47 (CH2) ---")
if oxidized_w47_ch2:
    oxidized_search_ch2 = gemmi.NeighborSearch(oxidized_gemmi_st, RADIUS_LIMIT)
    oxidized_neighbors_ch2 = list(oxidized_search_ch2.find_neighbors(oxidized_w47_ch2, max_dist=RADIUS_LIMIT))
    if oxidized_neighbors_ch2:
        for neighbor in oxidized_neighbors_ch2:
            atom = neighbor.atom
            if atom and atom.residue:
                print(f"  Atom: {atom.name}, Residue: {atom.residue.name}{atom.residue.seqid.num} Chain: {atom.residue.chain.name}, Distance: {neighbor.dist:.3f} Å")
                all_proximity_data.append({
                    'Structure Type': 'Oxidized',
                    'Target Atom': 'CH2',
                    'Target Residue': 'W47',
                    'Neighbor Atom': atom.name,
                    'Neighbor Residue': f"{atom.residue.name}{atom.residue.seqid.num}",
                    'Neighbor Chain': atom.residue.chain.name,
                    'Distance': neighbor.dist
                })
    else:
        print(f"  No neighbors found within {RADIUS_LIMIT}Å of Oxidized W47 CH2.")
else:
    print("  Oxidized W47 CH2 atom not found.")

# --- Proximity Analysis for Oxidized W47 (OXT) ---
oxidized_w47_oxt = find_atom_in_structure(oxidized_gemmi_st, 'A', 47, 'OXT')

print("\n--- Neighbors for Oxidized W47 (OXT) ---")
if oxidized_w47_oxt:
    oxidized_search_oxt = gemmi.NeighborSearch(oxidized_gemmi_st, RADIUS_LIMIT)
    oxidized_neighbors_oxt = list(oxidized_search_oxt.find_neighbors(oxidized_w47_oxt, max_dist=RADIUS_LIMIT))
    if oxidized_neighbors_oxt:
        for neighbor in oxidized_neighbors_oxt:
            atom = neighbor.atom
            if atom and atom.residue:
                print(f"  Atom: {atom.name}, Residue: {atom.residue.name}{atom.residue.seqid.num} Chain: {atom.residue.chain.name}, Distance: {neighbor.dist:.3f} Å")
                all_proximity_data.append({
                    'Structure Type': 'Oxidized',
                    'Target Atom': 'OXT',
                    'Target Residue': 'W47',
                    'Neighbor Atom': atom.name,
                    'Neighbor Residue': f"{atom.residue.name}{atom.residue.seqid.num}",
                    'Neighbor Chain': atom.residue.chain.name,
                    'Distance': neighbor.dist
                })
    else:
        print(f"  No neighbors found within {RADIUS_LIMIT}Å of Oxidized W47 OXT.")
else:
    print("  Oxidized W47 OXT atom not found.")

# --- Save proximity data to Excel ---
if all_proximity_data:
    df_proximity = pd.DataFrame(all_proximity_data)
    output_excel_path = '/content/proximity_analysis_results.xlsx'
    df_proximity.to_excel(output_excel_path, index=False)
    print(f"\nProximity analysis results saved to: {output_excel_path}")
else:
    print("\nNo proximity data to save.")